# Day 2: Reranker Ablations and Bootstrap Confidence Intervals

This notebook runs the Day 2 publication-strengthening analyses described in `results/plans.md`:

- reranker ablations over the saved top-256 RefMet candidate pool
- bootstrap confidence intervals for MRR and HitRate@k
- paired bootstrap differences for the final reranker improvement
- external model comparison bootstrap where per-query prediction files are available

It does **not** re-encode transformer embeddings. It reuses the canonical artifacts from `results/external_lab_refmet_model_comparison_reproducible/`.

## Configuration

For the full paper run, leave `DAY2_MAX_QUERIES_PER_SPLIT = None`. For a quick smoke test, set it to a small number such as `1000` before running the next cell.

In [ ]:
# Full run settings. These can be changed before executing the runner.
DAY2_POOL_K = 256
DAY2_RERANK_BATCH_SIZE = 384
DAY2_MAX_QUERIES_PER_SPLIT = None  # Example smoke test: 1000
DAY2_BOOTSTRAP_N = 1000
DAY2_BOOTSTRAP_SEED = 17
DAY2_BOOTSTRAP_SPLIT = 'test'

print('Day 2 configuration loaded.')

## Run Day 2 Analysis

This executes `scripts/run_day2_reranker_ablations_bootstrap.py`. The script resolves the repository root automatically, so the notebook can be launched from the repo root or from the `notebooks/` folder.

In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [
    cwd / 'scripts' / 'run_day2_reranker_ablations_bootstrap.py',
    cwd.parent / 'scripts' / 'run_day2_reranker_ablations_bootstrap.py',
]
script_path = next((p for p in candidates if p.exists()), None)
if script_path is None:
    raise FileNotFoundError('Could not find scripts/run_day2_reranker_ablations_bootstrap.py')

print('Running:', script_path)
exec(compile(script_path.read_text(encoding='utf-8'), str(script_path), 'exec'))

## Expected Outputs

After completion, check these files:

- `results/external_lab_refmet_model_comparison_reproducible/reranker_ablation_metrics.csv`
- `results/external_lab_refmet_model_comparison_reproducible/reranker_ablation_per_query_ranks.csv`
- `results/external_lab_refmet_model_comparison_reproducible/reranker_ablation_plot.png`
- `results/bootstrap_confidence_intervals.csv`
- `results/bootstrap_metric_differences.csv`
- `results/bootstrap_metric_differences.png`
- `results/external_lab_refmet_model_comparison_reproducible/day2_ablation_bootstrap_summary.md`

Interpretation checklist:

1. Confirm whether `full_hybrid` improves over `embedding_only` with bootstrap intervals excluding zero.
2. Compare `lexical_only_best_weights`, `dense_plus_exact`, and `dense_plus_soft_lexical` to understand which feature groups drive the gain.
3. Use `HitRate@256` as candidate-pool recall, not reranker quality, because every ablation reorders the same top-256 pool.
4. Use the bootstrap difference file for paper-ready statements about statistical confidence.